In [118]:
import pandas as pd

In [119]:
import os
os.environ["OPENAI_API_KEY"] = "##"

In [120]:
#!pip install -q youtube-transcript-api langchain-community langchain-openai \
               #faiss-cpu tiktoken python-dotenv

In [121]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter,Language
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

***Step 1a - Indexing (Document Ingestion)***

In [122]:
from youtube_transcript_api import YouTubeTranscriptApi

video_id = "ad79nYk2keg"

# Instantiate the API class to get transcripts
api = YouTubeTranscriptApi()
transcript_list = api.list(video_id).find_transcript(["en"]).fetch()

# Access the text using attribute lookup instead of key subscripting
transcript = " ".join(chunk.text for chunk in transcript_list)
print(transcript)

picture this a machine that could organize your cupboard just as you like it or serve every member of the house a customized cup of coffee makes your day easier doesn't it these are the products of artificial intelligence but why use the term artificial intelligence well these machines are artificially incorporated with human-like intelligence to perform tasks as we do this intelligence is built using complex algorithms and mathematical functions but ai may not be as obvious as in the previous examples in fact ai is used in smartphones cars social media feeds video games banking surveillance and many other aspects of our daily life the real question is what does an ai do at its core here is a robot we built in our lab which is now dropped onto a field in spite of a variation in lighting landscape and dimensions of the field the ai robot must perform as expected this ability to react appropriately to a new situation is called generalized learning the robot is now at a crossroad one that

***Step 1b - Indexing (Text Splitting)***

In [123]:
spliter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [124]:
chunks=spliter.create_documents([transcript])


In [125]:
len(chunks)

6

**Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)**

In [126]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)

In [127]:
vector_store.index_to_docstore_id

{0: 'e25c12e6-e197-4bc9-af8a-5d12bb95d1ce',
 1: '45949a72-ad50-481f-868e-01e773cf7417',
 2: '3a11033d-8186-4c67-aa18-c37808217240',
 3: '23800192-4eba-4ca4-bdf9-25907fbbf0ee',
 4: '1e34b525-0f6b-4761-a8f8-8c59a7c752e3',
 5: 'a2167d21-8630-4d12-a0c2-2a8050ea1a94'}

In [128]:
vector_store.get_by_ids(["6c0cbbe4-ba19-4949-9f6c-e712f04f2d46"])

[]

In [129]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [130]:
retriever.invoke('What is deepmind')

[Document(id='23800192-4eba-4ca4-bdf9-25907fbbf0ee', metadata={}, page_content="from avengers is an ideal example of a strong ai that's because it's self-aware and eventually even develops emotions this makes the ai's response unpredictable you must be wondering well how is artificial intelligence different from machine learning and deep learning we saw what ai is machine learning is a technique to achieve ai and deep learning in turn is a subset of machine learning machine learning provides a machine with the capability to learn from data and experience through algorithms deep learning does this learning through ways inspired by the human brain this means through deep learning data and patterns can be better perceived ray kurzweil a well-known futurist predicts that by the year 2045 we would have robots as smart as humans this is called the point of singularity well that's not all in fact elon musk predicts that the human mind and body will be enhanced by ai implants which would make 

***Step 3 - Augmentation***

In [131]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [132]:
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      {context}\n      Question: {question}\n    ")

In [133]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [134]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"you in short do not forget to leave your answer to the quiz in the comment section below also like share and subscribe to our channel if you enjoyed this video stay tuned and keep learning [Music] you\n\nthat's not all in fact elon musk predicts that the human mind and body will be enhanced by ai implants which would make us partly cyborgs so here's a question for you which of the below ai projects don't exist yet a an ai robot with citizenship b a robot with a muscular skeletal system c ai that can read its owner's emotions d ain't that develops emotions over time give it a thought and leave your answers in the comment section below three lucky winners will receive amazon gift vouchers since the human brain is still a mystery it's no surprise that ai2 has a lot of unventured domains for now ai is built to work with humans and make our tasks easier however with the maturation of technology we can only wait and watch what the future of ai holds for us well that is artificial intelligen

In [135]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

**Step 4 - Generation**

In [136]:
llm = ChatOpenAI(model="gpt-4o", temperature=0.2)

In [137]:
llm.invoke(final_prompt)

AIMessage(content="I don't know.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 4, 'prompt_tokens': 681, 'total_tokens': 685, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_4c910a9acf', 'id': 'chatcmpl-EjQpfjIbYZzXpjWUHgEcKWt3nSQTK', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1d055-400a-74ad-b69e-866ae5720398-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 681, 'output_tokens': 4, 'total_tokens': 685, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Building a Chain

In [152]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel,RunnableBranch,RunnableLambda

In [153]:
from langchain_core.output_parsers import StrOutputParser

In [154]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [155]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [156]:
#parallel_chain.invoke('who is Demis')

In [157]:
parser=StrOutputParser()

In [158]:
main_chain=parallel_chain | prompt | llm| parser

In [162]:
main_chain.invoke('tell Ray Kurzweil')

"I don't know."